In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from preprocessing.dataset import load_dataset

filename = "cost_E5-5.data"
data = load_dataset(filename)

Dataset cost_E5-5.data cargado con 60000 muestras.


In [ ]:
from models.cpmp_transformer_v6 import CPMPTransformer

H = 7
C_dim = 3
X_dim = 3
d_model = 64
nhead = 4
num_layers = 4
ff_dim_multiplier = 2
dropout = 0.3

model = CPMPTransformer(
    H=H,
    C_dim=3,
    X_dim=X_dim,
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    ff_dim_multiplier=ff_dim_multiplier,
    dropout=dropout
)

In [4]:
from training.training import sl_train, Accuracy, CrossEntropyLoss, MSE

epochs = 100
train_size = 4000
test_size = 1000
batch_size = 64
learning_rate = 1e-4
weight_decay = 1e-4
patience = 10
seed = 42

loss_functions = [CrossEntropyLoss(), MSE()]
metrics = [[Accuracy()], []]

model = sl_train(model, epochs, data, train_size, test_size, batch_size, learning_rate, weight_decay, loss_functions, patience, metrics, seed)

ℹ️ Usando dispositivo: cpu


/home/oscar/Escritorio/CPMP-Framework/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Epoch 1/100
    Train CrossEntropy: 2.6441 | Train MSE: 185.8903
    Val CrossEntropy: 2.5902 | Val MSE: 178.1725
 | Accuracy: 17.50%
Epoch 2/100
    Train CrossEntropy: 2.5859 | Train MSE: 175.8960
    Val CrossEntropy: 2.5379 | Val MSE: 170.5376
 | Accuracy: 21.20%
Epoch 3/100
    Train CrossEntropy: 2.5495 | Train MSE: 170.6488
    Val CrossEntropy: 2.5309 | Val MSE: 167.9125
 | Accuracy: 20.40%
Epoch 4/100
    Train CrossEntropy: 2.5384 | Train MSE: 164.0461
    Val CrossEntropy: 2.5184 | Val MSE: 156.8006
 | Accuracy: 22.30%
Epoch 5/100
    Train CrossEntropy: 2.5232 | Train MSE: 155.8369
    Val CrossEntropy: 2.5089 | Val MSE: 148.7620
 | Accuracy: 23.30%
Epoch 6/100
    Train CrossEntropy: 2.5140 | Train MSE: 148.2994
    Val CrossEntropy: 2.5000 | Val MSE: 141.3977
 | Accuracy: 24.30%
Epoch 7/100
    Train CrossEntropy: 2.5036 | Train MSE: 140.6530
    Val CrossEntropy: 2.4940 | Val MSE: 135.3560
 | Accuracy: 23.00%
Epoch 8/100
    Train CrossEntropy: 2.4963 | Train MSE: 134.0

In [3]:
from models.cost.cost_predictor import CostPredictorTransformer

H = 7
C_dim = 3
X_dim = 3
d_model = 64
nhead = 4
num_layers = 4
ff_dim_multiplier = 2
dropout = 0.3

model = CostPredictorTransformer(
    H=H,
    C_dim=3,
    X_dim=X_dim,
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    ff_dim_multiplier=ff_dim_multiplier,
    dropout=dropout
)

In [ ]:
from training.training import sl_train, MSE, ExpMSE, ExpMAE

epochs = 100
train_size = 45000
test_size = 15000
batch_size = 64
learning_rate = 1e-4
weight_decay = 1e-4
patience = 10
seed = 42

loss_functions = [MSE()]
metrics = [[ExpMSE(), ExpMAE()]]

model = sl_train(model, epochs, data, train_size, test_size, batch_size, learning_rate, weight_decay, loss_functions, patience, metrics, seed)

ℹ️ Usando dispositivo: cpu

Epoch 1/100
    Train MSE: 0.1612
    Val MSE: 0.1313
 | ExpMSE: 16.5388 | ExpMAE: 3.2617


KeyboardInterrupt: 

In [5]:
from training.training import save_model

save_model(model, "multi")

✅ Modelo guardado en /home/oscar/Escritorio/CPMP-Framework/models/multi.pth


In [26]:
from training.training import load_model, rl_train, CrossEntropyLoss, Accuracy, MSE, DataGenerationConfigRL
from generation.adapters import EnrichedStackMatrixAdapter, StackMatrix4D3FAdapter, ExtraDataAdapter3F, MultiOutputAdapter
from models.cpmp_transformer_v9 import CPMPTransformer

model = load_model(CPMPTransformer, "multi_rl5")

iterations = 10
epochs = 20
train_size = 2400
test_size = 600
batch_size = 16
learning_rate = 5e-5
weight_decay = 1e-4
patience = 5
seed = 42

loss_functions = [CrossEntropyLoss(), MSE()]
metrics = [[Accuracy()], []]

datagen_config = DataGenerationConfigRL(
    instance_sets=["E5-5-R20-RL", "E5-5-R50-RL", "E5-5-R100-RL"],
    H=7,
    max_steps=50,
    layout_adapter_config=(EnrichedStackMatrixAdapter, StackMatrix4D3FAdapter, ExtraDataAdapter3F),
    moves_adapter_config=(MultiOutputAdapter, ),
    num_workers=12
)

model = rl_train(model, iterations, datagen_config, epochs, train_size, test_size, batch_size, learning_rate, weight_decay, loss_functions, patience, metrics, seed)

ℹ️ Usando dispositivo: cpu
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/tmp_train.data (Tamaño 2397)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/tmp_test.data (Tamaño 600)
Tamaño datasets | Train: 2397 | Test: 600
Costo promedio | Train: 12.41 | Test: 12.53

Epoch 1/20
    Train CrossEntropy: 1.9135 | Train MSE: 1.4341
    Val CrossEntropy: 1.9199 | Val MSE: 1.4061
 | Accuracy: 54.33%
Epoch 2/20
    Train CrossEntropy: 1.9090 | Train MSE: 1.4096
    Val CrossEntropy: 1.9209 | Val MSE: 1.4206
 | Accuracy: 54.50%
Epoch 3/20
    Train CrossEntropy: 1.8895 | Train MSE: 1.3743
    Val CrossEntropy: 1.9230 | Val MSE: 1.3807
 | Accuracy: 54.67%
Epoch 4/20
    Train CrossEntropy: 1.8888 | Train MSE: 1.3485
    Val CrossEntropy: 1.9176 | Val MSE: 1.3963
 | Accuracy: 55.17%
Epoch 5/20
    Train CrossEntropy: 1.8688 | Train MSE: 1.3849
    Val CrossEntropy: 1.9256 | Val MSE: 1.3368
 | Accuracy: 54.67%
Epoch 6/20
    Train CrossEntropy: 1.8637 | Train MSE: 1.37

In [27]:
from training.training import save_model

save_model(model, "multi_rl6")

✅ Modelo guardado en /home/oscar/Escritorio/CPMP-Framework/models/multi_rl6.pth


In [1]:
from training.training import load_model, rl_train, Accuracy, DataGenerationConfigRL
from generation.adapters import EnrichedStackMatrixAdapter, StackMatrix4D3FAdapter, ExtraDataAdapter3F, DefaultMovesAdapter
from models.cpmp_transformer_v5 import CPMPTransformer

model = load_model(CPMPTransformer, "best_v5")

iterations = 1
epochs = 2
train_size = 100
test_size = 100
batch_size = 128
learning_rate = 5e-5
weight_decay = 1e-4
patience = 5
metrics = [Accuracy()]
seed = 42

datagen_config = DataGenerationConfigRL(
    instance_sets=["E5-25-43", "E5-25-84"],
    H=7,
    max_steps=50,
    layout_adapter_config=(EnrichedStackMatrixAdapter, StackMatrix4D3FAdapter, ExtraDataAdapter3F),
    moves_adapter_config=(DefaultMovesAdapter, ),
    num_workers=12
)

model = rl_train(model, iterations, datagen_config, epochs, train_size, test_size, batch_size, learning_rate, weight_decay, patience, metrics, seed)

ℹ️ Usando dispositivo: cpu
Tamaño datasets | Train: 100 | Test: 100
Costo promedio | Train: 14.47 | Test: 13.40

Epoch 1/2
    Average - Train Loss: 2.0875 | Val Loss: 2.0537
 | Accuracy: 48.00%


KeyboardInterrupt: 

In [3]:
from training.training import save_model

save_model(model, "rl3")

✅ Modelo guardado en /home/oscar/Escritorio/CPMP-Framework/models/rl3.pth
